# Actual EMRIO Checkpoint Verification

This notebook verifies selected aggregate, non-reversible checkpoints from the actual EMRIO calculation pipeline. It reads only audited CSV files from `public_data/checkpoints/` and presents compact tables and numerical checks rather than visual summaries. The goal is to let reviewers inspect the public-safe checkpoint outputs without exposing licensed inputs, firm-level rows, or reversible matrices.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

CHECKPOINT_DIR = Path("../public_data/checkpoints")
assert CHECKPOINT_DIR.exists(), f"Missing checkpoint directory: {CHECKPOINT_DIR}"

def fmt_value(x):
    if pd.isna(x):
        return ""
    try:
        x = float(x)
    except Exception:
        return x
    if abs(x) >= 1e6 or (abs(x) > 0 and abs(x) < 1e-4):
        return f"{x:.3e}"
    if abs(x - round(x)) < 1e-9:
        return f"{x:.0f}"
    return f"{x:.6f}"

## Load Checkpoint Tables

The table below lists the audited public checkpoint files used by this notebook. File sizes are intentionally small because the files contain only aggregate, non-reversible outputs.


In [2]:
checkpoint_files = {
    "sector_emissions": "01_sector_emissions_by_country_sic.csv",
    "scope1_allocation": "02_scope1_allocation_summary_by_country_sic.csv",
    "emrio_diagnostics": "03_emrio_aggregation_diagnostics.csv",
    "scope23": "04_scope23_by_country_sic.csv",
    "reported_country": "05_reported_benchmark_by_country.csv",
    "reported_sic": "06_reported_benchmark_by_sic.csv",
    "estimators": "07_estimator_definitions_summary.csv",
    "open_mrio": "08_open_mrio_sic_comparison.csv",
    "uncertainty": "09_uncertainty_summary.csv",
    "representativeness": "10_representativeness_summary.csv",
}

tables = {name: pd.read_csv(CHECKPOINT_DIR / filename) for name, filename in checkpoint_files.items()}
manifest = []
for name, filename in checkpoint_files.items():
    path = CHECKPOINT_DIR / filename
    df = tables[name]
    manifest.append({
        "checkpoint": name,
        "file": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "size_kb": round(path.stat().st_size / 1024, 2),
    })
manifest = pd.DataFrame(manifest)
display(manifest)


,checkpoint,file,rows,columns,size_kb
0,sector_emissions,01_sector_emissions_by_country_sic.csv,117,7,10.22
1,scope1_allocation,02_scope1_allocation_summary_by_country_sic.csv,125,7,13.36
2,emrio_diagnostics,03_emrio_aggregation_diagnostics.csv,12,3,0.94
3,scope23,04_scope23_by_country_sic.csv,125,11,23.50
4,reported_country,05_reported_benchmark_by_country.csv,24,7,1.90
5,reported_sic,06_reported_benchmark_by_sic.csv,9,7,0.99
6,estimators,07_estimator_definitions_summary.csv,7,3,0.69
7,open_mrio,08_open_mrio_sic_comparison.csv,30,6,2.88
8,uncertainty,09_uncertainty_summary.csv,3,13,0.89
9,representativeness,10_representativeness_summary.csv,2,8,0.60


## 1. Sector-Emission Construction Checkpoint

This checkpoint summarizes the actual sector-emission construction step at country and sector-prefix level. It is derived from the Step 1 country-sector output, after aggregation.


In [3]:
sector = tables["sector_emissions"]
sector_country = (
    sector.groupby("country", as_index=False)
    .agg(
        n_source_sectors=("n_source_sectors", "sum"),
        sector_output_total=("sector_output_total", "sum"),
        ghg6_total=("ghg6_total", "sum"),
    )
    .sort_values("ghg6_total", ascending=False)
)

display(
    sector_country.head(12).style.format({
        "n_source_sectors": fmt_value,
        "sector_output_total": fmt_value,
        "ghg6_total": fmt_value,
    })
)

sector_summary = pd.DataFrame({
    "diagnostic": ["countries", "source sectors after aggregation", "aggregate GHG6 total"],
    "value": [sector_country["country"].nunique(), sector_country["n_source_sectors"].sum(), sector_country["ghg6_total"].sum()],
})
display(sector_summary.style.format({"value": fmt_value}))

,country,n_source_sectors,sector_output_total,ghg6_total
69,USA,405,3.151e+07,6.287e+09
36,IND,66,3.937e+06,3.152e+09
61,RUS,59,2.402e+06,2.175e+09
40,JPN,395,8.410e+06,1.292e+09
10,BRA,67,3.118e+06,1.284e+09
20,DEU,63,6.110e+06,9.032e+08
51,MEX,262,2.127e+06,7.782e+08
11,CAN,236,2.751e+06,7.423e+08
42,KOR,30,3.249e+06,7.219e+08
2,AUS,114,2.324e+06,5.881e+08


,diagnostic,value
0,countries,71
1,source sectors after aggregation,5621
2,aggregate GHG6 total,2.479e+10


## 2. Scope 1 Allocation Summary Checkpoint

This checkpoint shows aggregate Scope 1 allocation summaries by country and SIC division. It does not include firm or segment rows.


In [4]:
scope1 = tables["scope1_allocation"]
scope1_sic = (
    scope1.groupby("sic_division", as_index=False)
    .agg(n_segments=("n_segments", "sum"), scope1_total=("scope1_total", "sum"), sales_total=("sales_total", "sum"))
    .sort_values("scope1_total", ascending=False)
)
scope1_sic["scope1_per_sales"] = scope1_sic["scope1_total"] / scope1_sic["sales_total"]

display(
    scope1_sic.style.format({
        "n_segments": fmt_value,
        "scope1_total": fmt_value,
        "sales_total": fmt_value,
        "scope1_per_sales": fmt_value,
    })
)

,sic_division,n_segments,scope1_total,sales_total,scope1_per_sales
9,"Transportation, Communications, Electric, Gas and Sanitary service",565,1.025e+09,1.428e+06,717.974483
3,Manufacturing,2487,8.599e+08,5.248e+06,163.863415
4,Mining,202,1.807e+08,314638.377784,574.465847
5,Other / suppressed small cells,91,6.423e+07,239635.983280,268.028930
10,Wholesale Trade,173,3.807e+07,518532.590281,73.421441
7,Retail Trade,339,1.517e+07,1.175e+06,12.904119
8,Services,518,1.082e+07,729498.295408,14.832341
2,"Finance, Insurance and Real Estate",573,6.374e+06,1.126e+06,5.658579
1,Construction,96,2.757e+06,148891.634200,18.515475
0,"Agriculture, Forestry and Fishing",10,1.388e+06,18302.289780,75.822569


## 3. EMRIO Aggregation Diagnostics

This checkpoint reports non-reversible diagnostics from the actual EMRIO aggregation and Scope calculation outputs. It releases counts and aggregate consistency checks, not the transaction matrix.


In [5]:
diagnostics = tables["emrio_diagnostics"]
display(diagnostics)

scope_check = diagnostics.loc[diagnostics["diagnostic"] == "scope23_minus_scope2_minus_scope3_abs", "value"].iloc[0]
print(f"Scope residual consistency check: {scope_check:.6g}")
assert abs(scope_check) < 1e-3


,diagnostic,value,note
0,q_est_rows,3.080900e+04,Rows in private q_est output; no row-level data released here.
1,q_est_countries,7.100000e+01,Unique countries represented in q_est.
2,q_est_sector_codes,1.229000e+03,Unique sector codes represented in q_est.
3,named_rows,1.965100e+04,Rows not classified as residual Other Source accounts.
4,other_source_rows,1.115800e+04,Residual Other Source rows identified by Others- prefix.
5,figure_segment_rows,5.095000e+03,Rows in private figure segment source; only count is released.
6,figure_countries,3.600000e+01,Unique countries in private figure segment source.
7,scope1_total,2.489253e+10,Aggregate Scope 1 total from q_est.
8,scope2_total,5.111031e+09,Aggregate Scope 2 total from q_est.
9,scope23_total,4.077765e+10,Aggregate Scope 2+3 total from q_est.


Scope residual consistency check: 0.000236511


## 4. Scope 2 / Scope 3 Summary Checkpoint

This checkpoint shows aggregate Scope 1, 2, and 3 totals and shares. Scope 3 is the residual after extracting Scope 2 from upstream Scope 2+3.


In [6]:
scope23 = tables["scope23"]
scope23_sic = (
    scope23.groupby("sic_division", as_index=False)
    .agg(
        n_segments=("n_segments", "sum"),
        scope1_total=("scope1_total", "sum"),
        scope2_total=("scope2_total", "sum"),
        scope3_total=("scope3_total", "sum"),
    )
)
scope23_sic["scope123_total"] = scope23_sic[["scope1_total", "scope2_total", "scope3_total"]].sum(axis=1)
for col in ["scope1", "scope2", "scope3"]:
    scope23_sic[f"share_{col}"] = scope23_sic[f"{col}_total"] / scope23_sic["scope123_total"]

scope_share_table = scope23_sic[["sic_division", "n_segments", "share_scope1", "share_scope2", "share_scope3"]].sort_values(
    "share_scope3", ascending=False
)
display(
    scope_share_table.style.format({
        "n_segments": fmt_value,
        "share_scope1": "{:.3f}",
        "share_scope2": "{:.3f}",
        "share_scope3": "{:.3f}",
    })
)

scope_total_check = pd.DataFrame({
    "metric": ["Scope 1", "Scope 2", "Scope 3", "Scope 1+2+3"],
    "total": [
        scope23_sic["scope1_total"].sum(),
        scope23_sic["scope2_total"].sum(),
        scope23_sic["scope3_total"].sum(),
        scope23_sic["scope123_total"].sum(),
    ],
})
display(scope_total_check.style.format({"total": fmt_value}))

,sic_division,n_segments,share_scope1,share_scope2,share_scope3
1,Construction,96,0.049,0.018,0.932
2,"Finance, Insurance and Real Estate",573,0.038,0.038,0.925
8,Services,518,0.070,0.089,0.841
7,Retail Trade,339,0.047,0.139,0.814
10,Wholesale Trade,173,0.166,0.053,0.782
6,Public Administration,41,0.253,0.031,0.716
0,"Agriculture, Forestry and Fishing",10,0.196,0.091,0.713
3,Manufacturing,2487,0.257,0.053,0.691
5,Other / suppressed small cells,91,0.331,0.031,0.638
9,"Transportation, Communications, Electric, Gas and Sanitary service",565,0.570,0.035,0.395


,metric,total
0,Scope 1,2.206e+09
1,Scope 2,3.486e+08
2,Scope 3,4.060e+09
3,Scope 1+2+3,6.614e+09


## 5. Reported--Benchmark Comparison Checkpoints

These checkpoints summarize the reported-versus-benchmark comparison by country and SIC division. Secondary binary subgroup counts are masked when either side of the binary split is too small.


In [7]:
reported_country = tables["reported_country"]
reported_sic = tables["reported_sic"]

country_cols = [
    "country", "n_reporting_companies", "benchmark_scope3_total", "reported_scope3_total",
    "benchmark_to_reported_ratio", "share_benchmark_exceeds_report",
]
display(
    reported_country.sort_values("n_reporting_companies", ascending=False).head(12)[country_cols]
    .style.format({
        "n_reporting_companies": fmt_value,
        "benchmark_scope3_total": fmt_value,
        "reported_scope3_total": fmt_value,
        "benchmark_to_reported_ratio": fmt_value,
        "share_benchmark_exceeds_report": fmt_value,
    })
)

display(
    reported_sic.sort_values("benchmark_to_reported_ratio", ascending=False)
    .style.format({
        "n_reporting_companies": fmt_value,
        "benchmark_scope3_total": fmt_value,
        "reported_scope3_total": fmt_value,
        "benchmark_to_reported_ratio": fmt_value,
        "share_benchmark_exceeds_report": fmt_value,
    })
)

,country,n_reporting_companies,benchmark_scope3_total,reported_scope3_total,benchmark_to_reported_ratio,share_benchmark_exceeds_report
22,USA,91,3.772e+08,2.909e+08,1.296492,0.780220
15,JPN,78,2.825e+08,1.001e+08,2.822649,0.923077
11,FRA,42,1.718e+08,8.504e+07,2.020291,0.785714
12,GBR,36,2.696e+08,6.536e+07,4.124707,0.750000
7,DEU,35,2.750e+08,1.519e+08,1.810786,0.771429
4,CAN,23,5.726e+07,7.525e+06,7.609604,
3,BRA,22,1.125e+08,2.573e+07,4.371007,
9,ESP,21,1.293e+08,3.373e+07,3.832219,1
10,FIN,20,4.545e+07,2.299e+07,1.976379,0.800000
19,SWE,16,1.786e+07,1.313e+07,1.360561,


,sic_division,n_reporting_companies,benchmark_scope3_total,reported_scope3_total,n_benchmark_exceeds_report,benchmark_to_reported_ratio,share_benchmark_exceeds_report
1,"Finance, Insurance and Real Estate",72,1.129e+08,5.449e+06,68.000000,20.708757,0.944444
2,Services,48,7.613e+07,1.631e+07,nan,4.667657,
0,Mining,22,1.047e+08,3.095e+07,nan,3.383029,
8,Retail Trade,19,1.385e+08,4.506e+07,13.000000,3.074954,0.684211
4,"Transportation, Communications, Electric, Gas and Sanitary service",72,3.498e+08,1.527e+08,65.000000,2.291070,0.902778
5,Unknown,13,7.168e+07,3.184e+07,nan,2.250958,
7,Construction,15,3.202e+07,1.527e+07,nan,2.097311,
6,Wholesale Trade,8,4.736e+07,4.123e+07,4.000000,1.148589,0.500000
3,Manufacturing,236,1.243e+09,1.107e+09,187.000000,1.122937,0.792373


## 6. Estimator Definitions

The estimator checkpoint records the headline ratio-of-aggregate-totals and alternative firm-ratio summaries used to interpret the reported--benchmark comparison.


In [8]:
estimators = tables["estimators"]
display(estimators)
headline = estimators.loc[estimators["metric"] == "ratio_of_aggregate_totals", "value"].iloc[0]
print(f"Headline ratio of aggregate totals: {headline:.3f}x")


,metric,value,note
0,n_reporting_companies,5.070000e+02,Number of company-level observations used in Figure 1 denominator comparison.
1,aggregate_benchmark_scope3,2.182848e+09,Aggregate EMRIO-based Scope 3 benchmark for the comparison set.
2,aggregate_reported_scope3,1.447378e+09,Aggregate company-reported Scope 3 for the comparison set.
3,ratio_of_aggregate_totals,1.508140e+00,Headline estimator used in the manuscript.
4,mean_firm_level_ratio,1.464521e+02,Mean of firm-level benchmark/reported ratios.
5,median_firm_level_ratio,6.394594e+00,Median of firm-level benchmark/reported ratios.
6,share_benchmark_exceeds_report,8.481262e-01,Share of companies for which benchmark exceeds report.


Headline ratio of aggregate totals: 1.508x


## 7. Open-MRIO Reference Checkpoint

This checkpoint compares SIC-level Scope shares across EMRIO, EXIOBASE, and GLORIA. The open-MRIO values are structural references, not exact numerical validation targets.


In [9]:
open_mrio = tables["open_mrio"]
display(open_mrio.groupby("database").size().rename("rows").reset_index())

pivot_s3 = open_mrio.pivot(index="sic_division", columns="database", values="share_S3")
first_col = pivot_s3.columns[0]
pivot_s3 = pivot_s3.sort_values(first_col, ascending=False)
display(pivot_s3.style.format("{:.3f}"))

scope_share_ranges = (
    open_mrio.groupby("database")[["share_S1", "share_S2", "share_S3"]]
    .agg(["min", "median", "max"])
)
display(scope_share_ranges.style.format("{:.3f}"))

,database,rows
0,EMRIO,10
1,EXIOBASE v3.9.6,10
2,GLORIA v060,10


database,EMRIO,EXIOBASE v3.9.6,GLORIA v060
sic_division,,,
Construction,0.935,0.901,0.885
"Finance, Insurance and Real Estate",0.926,0.634,0.688
Services,0.838,0.729,0.833
Retail Trade,0.819,0.756,0.814
Wholesale Trade,0.779,0.787,0.786
"Agriculture, Forestry and Fishing",0.728,0.356,0.293
Public Administration,0.718,0.709,0.745
Manufacturing,0.692,0.682,0.707
Mining,0.417,0.326,0.421


## 8. Uncertainty and Representativeness Checkpoints

These checkpoints show aggregate Monte Carlo holdout intervals and aggregate representativeness summaries. They do not contain company-level draws or company-level coverage records.


In [10]:
uncertainty = tables["uncertainty"]
representativeness = tables["representativeness"]

display(uncertainty.style.format({col: fmt_value for col in uncertainty.select_dtypes(include="number").columns}))
display(representativeness.style.format({col: fmt_value for col in representativeness.select_dtypes(include="number").columns}))

,level,year,metric,point_estimate_original,ui95_low,ui95_mid,ui95_high,ui90_low,ui90_high,relative_width_95,n_iter,interval_type,uncertainty_source
0,total,2015,g_S2_EMRIO,5.111e+09,5.111e+09,5.111e+09,5.111e+09,5.111e+09,5.111e+09,2.045e-05,30,simulation_based_95pct_uncertainty_interval,replication_based_emrio_iter_recentered
1,total,2015,g_S23_EMRIO,4.078e+10,4.076e+10,4.078e+10,4.081e+10,4.076e+10,4.081e+10,0.001221,30,simulation_based_95pct_uncertainty_interval,replication_based_emrio_iter_recentered
2,total,2015,g_S3_T,3.567e+10,3.565e+10,3.567e+10,3.570e+10,3.565e+10,3.570e+10,0.001392,30,simulation_based_95pct_uncertainty_interval,replication_based_emrio_iter_recentered


,sample,n_rows,numerator_metric,numerator_value,denominator_metric,denominator_value,coverage_share,note
0,matched_company_rows_available,2012,scope1_plus_scope2plus3,6.768e+09,full_emrio_scope1_plus_scope2plus3,6.567e+10,0.103059,"Computed from public checkpoint generation inputs; row count reflects available df_c output, not a released firm table."
1,reported_denominator_comparison,507,scope3_benchmark,2.183e+09,full_emrio_scope3,3.567e+10,0.061201,Computed from reported-denominator comparison benchmark total relative to full EMRIO Scope 3 total.


## 9. Public-Release Audit Checks

The checks below are intentionally conservative. They verify that checkpoint columns and values do not expose restricted identifiers, local paths, invalid numeric values, or unmasked small binary subgroups in the reported--benchmark tables.


In [11]:
forbidden_patterns = [
    "FACT" + "SET",
    "ENT" + "ITY",
    "PRO" + "PER",
    "ACCOUNT" + "_NAME",
    "IS" + "IN",
    "/" + "mnt/",
    "/" + "home/",
]

for name, df in tables.items():
    joined_columns = " ".join(map(str, df.columns)).upper()
    for pattern in forbidden_patterns:
        assert pattern.upper() not in joined_columns, (name, pattern, df.columns.tolist())
    string_values = " ".join(str(v) for v in df.select_dtypes(include="object").fillna("").to_numpy().ravel())
    for pattern in ["/" + "mnt/", "/" + "home/"]:
        assert pattern not in string_values, (name, pattern)
    numeric = df.select_dtypes(include=[np.number])
    assert not np.isinf(numeric.to_numpy()).any(), name

for name in ["reported_country", "reported_sic"]:
    df = tables[name]
    visible = df.dropna(subset=["n_benchmark_exceeds_report"])
    for _, row in visible.iterrows():
        n = int(row["n_reporting_companies"])
        exceeds = int(row["n_benchmark_exceeds_report"])
        complement = n - exceeds
        assert not (0 < exceeds <= 3), (name, row.iloc[0], n, exceeds, complement)
        assert not (0 < complement <= 3), (name, row.iloc[0], n, exceeds, complement)

print("Public checkpoint audit checks passed.")


Public checkpoint audit checks passed.


## What This Notebook Does Not Verify

This notebook does not reconstruct the licensed firm-to-firm transaction matrix, firm-segment mappings, matched company records, or reversible intermediate matrices. Exact end-to-end execution from raw inputs requires licensed data access. The notebook verifies the public aggregate checkpoints and the continuity of the published calculation path at a non-reversible aggregation level.
